In [9]:
from openai import OpenAI
from typing import Literal
from pathlib import Path
from pydantic import BaseModel
from dotenv import load_dotenv
from time import time
import json
import os

Definido Modelo LLM Local

In [3]:
load_dotenv(dotenv_path=Path(".env"))

MODEL = "sabiazinho-4"

client = OpenAI(
    base_url="https://chat.maritaca.ai/api",
    api_key=os.getenv("MARITACA_API_KEY"),
)

SCHEMAS

In [4]:
class ToolScore(BaseModel):
    tool: Literal["cag", "rag", "mongo"]
    expected_value: float
    estimated_cost: float
    reason: str


class GraphPlan(BaseModel):
    selected_tools: list[Literal["cag", "rag", "mongo"]]
    scores: list[ToolScore]
    reason: str


class JudgeDecision(BaseModel):
    is_enough: bool
    reason: str
    missing_info: str | None = None

In [5]:
graph_state = {
    "question": None,
    "plan": None,
    "tool_results": [],
    "aggregated_context": None,
    "judge": None,
    "final_answer": None,
    "logs": [],
}

In [6]:
def extract_json(raw_text: str) -> dict:
    json_start = raw_text.find("{")
    json_end = raw_text.rfind("}")

    if json_start == -1 or json_end == -1:
        raise ValueError(f"Resposta sem JSON valido: {raw_text}")

    return json.loads(raw_text[json_start:json_end + 1])


In [ ]:
def plan_node(question: str) -> GraphPlan:
    prompt = f"""
PAPEL:
  Voce e o PLANNER de um sistema chamado Maritaca Hybrid Graph Agentic.

OBJETIVO:
Escolher quais ferramentas devem ser usadas para responder a pergunta do usuario.

CONTEXTO:
O sistema possui tres memorias/ferramentas:
- CAG para conhecimento cacheado.
- RAG para documentos e evidencias textuais.
- Mongo para dados estruturados e registros vivos.

OPCOES DISPONIVEIS:
- cag
- rag
- mongo

REGRAS:
- Se a pergunta for simples e ligada a FAQ, regras, horarios, planos ou precos, escolha ["cag"].
- Se a pergunta pedir documentos, contratos, PDFs, manuais ou evidencias textuais, escolha ["rag"].
- Se a pergunta pedir clientes, pedidos, produtos, usuarios ou registros, escolha ["mongo"].
- Se a pergunta precisar combinar mais de uma fonte, escolha mais de uma ferramenta.
- Nao escolha todas as ferramentas sem necessidade.
- Sempre gere score para cag, rag e mongo.
- expected_value deve ir de 0.0 a 1.0.
- estimated_cost deve ir de 0.0 a 1.0.

ENTRADA:
Pergunta do usuario:
{question}

FORMATO DE SAIDA:
Responda APENAS JSON valido.
Nao use markdown.
Nao escreva texto antes ou depois do JSON.
O JSON deve ter exatamente este formato:

{{
  "selected_tools": ["cag"],
  "scores": [
    {{
      "tool": "cag",
      "expected_value": 0.9,
      "estimated_cost": 0.1,
      "reason": "motivo curto"
    }},
    {{
      "tool": "rag",
      "expected_value": 0.2,
      "estimated_cost": 0.4,
      "reason": "motivo curto"
    }},
    {{
      "tool": "mongo",
      "expected_value": 0.1,
      "estimated_cost": 0.3,
      "reason": "motivo curto"
    }}
  ],
  "reason": "motivo geral da selecao"
}}
"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
    )

    raw_text = response.choices[0].message.content
    data = extract_json(raw_text)
    plan = GraphPlan(**data)

    graph_state["logs"].append({
        "node": "plan_node",
        "elapsed": round(time.time() - start, 2),
        "input": {
            "question": question,
        },
        "output": plan.model_dump(),
    })

    return plan


In [ ]:
def judge_node(question: str, aggregated_context: str) -> JudgeDecision:
    start = time.time()


    prompt = f"""
PAPEL:
Voce e o JUDGE de um sistema chamado Maritaca Hybrid Graph Agentic.

OBJETIVO:
Avaliar se o contexto é suficiente diante da pergunta do usuario.

CONTEXTO:
O sistema possui tres memorias/ferramentas:
- CAG para conhecimento cacheado.
- RAG para documentos e evidencias textuais.
- Mongo para dados estruturados e registros vivos.

OPCOES DISPONIVEIS:
Voce só deve justificar com:
- is_enough true = se o contexto permite responder o usuário com seguranca.
- is_enough false = se ainda falta inforacao importante.

REGRAS:
- Seja conservador: se faltar dado essencial, marque is_enough como false.
- Nao invente informacoes que nao estejam no contexto.
- Se a pergunta pede dados de cliente e o contexto nao tem dados do cliente, marque false.
- Se a pergunta pede documento/contrato e o contexto nao tem evidencias documentais, marque false.
- Se a resposta esta claramente no contexto, marque true.
- missing_info deve ser null quando is_enough for true.
- missing_info deve explicar o que falta quando is_enough for false.

ENTRADA:
Pergunta do usuario:
{question}

Contexto agregado:
{aggregated_context}

FORMATO DE SAIDA:
Responda APENAS JSON valido.
O JSON deve ter exatamente este formato:

{{
  "is_enough": true,
  "reason": "O contexto contem a informacao necessaria para responder.",
  "missing_info": null
}}
"""


    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
    )

    raw_text = response.choices[0].message.content
    data = extract_json(raw_text)

    judge = JudgeDecision(**data)

    graph_state["logs"].append({
        "node": "judge_node",
        "elapsed": round(time.time() - start, 2),
        "input": {
            "question": question,
            "aggregated_context": aggregated_context,
        },        
        "output": judge.model_dump(),
    })

    return judge


In [ ]:
def final_node(question: str, aggregated_context: str) -> str:
    start = time.time()

    prompt = f"""
PAPEL: 
Voce e o Decisor Final.

OBJETIVO:
Voce tem como objetivo responder a pergunta que o cliente realizou usando apenas o contexto fornecido.

CONTEXTO:
{aggregated_context}

REGRAS:
- Responda em portugues do Brasil.
- Use apenas informacoes presentes no contexto.
- Nao invente informacoes.
- Se o contexto nao tiver informacao suficiente, diga isso claramente.
- Seja claro e direto.

ENTRADA:
Pergunta do Usuario:
{question}

FORMATO DE SAIDA:
Responda em texto normal, sem JSON ou markdown.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
    )
    
    
    raw_text = response.choices[0].message.content

    graph_state["logs"].append({
        "node": "final_node",
        "elapsed": round(time.time() - start, 2),
        "input": {
            "question": question,
            "aggregated_context": aggregated_context,
        },
        "output": raw_text,
    })

    return raw_text
